In [1]:
from pyspark.sql import SparkSession
from graphframes import GraphFrame
import pyspark.sql.functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType, FloatType, LongType, BooleanType, DoubleType

spark = SparkSession.builder \
    .appName("GraphFramesPySpark4") \
    .master("local[*]") \
    .config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.10.1") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/25 18:00:24 WARN Utils: Your hostname, cubone, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlan0)
26/05/25 18:00:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/massimiliano/Work/DistArch/Spark/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/massimiliano/.ivy2.5.2/cache
The jars for the packages stored in: /home/massimiliano/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d2310ebd-c164-4cdd-8058-c3e20384f236;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.11.0 in central
	found io.graphframes#graphframes-graphx-spark4_2.13;0.11.0 in central
:: resolution report :: resolve 57ms :: artifact

In [2]:
airlines = spark.read.load("data/airlines.csv",\
                     format="csv",\
                     header=True,\
                     inferSchema=True)
airlines.printSchema()

airports = spark.read.load("data/airports.csv",\
                     format="csv",\
                     header=True,\
                     inferSchema=True)
airports.printSchema()

routes = spark.read.load("data/routes.csv", \
                    format="csv",\
                    header=True,\
                    inferSchema=True)
routes.printSchema()

root
 |-- airline_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- alias: string (nullable = true)
 |-- iata: string (nullable = true)
 |-- icao: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- country: string (nullable = true)
 |-- active: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- iata: string (nullable = true)
 |-- icao: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- altitude: integer (nullable = true)
 |-- timezone: string (nullable = true)
 |-- DST: string (nullable = true)
 |-- tz_timezone: string (nullable = true)
 |-- type: string (nullable = true)
 |-- source: string (nullable = true)

root
 |-- airline_iata: string (nullable = true)
 |-- airline_id: integer (nullable = true)
 |-- airport_source: string (nullable = true)
 |-- airport_

In [3]:
vertices_clean = airports.withColumn("id", airports.id.cast("string"))

edges_clean = routes \
    .withColumnRenamed("airport_source_id", "src") \
    .withColumnRenamed("airport_destination_id", "dst") \
    .filter("src IS NOT NULL AND dst IS NOT NULL") \
    .withColumn("src", F.col("src").cast("string")) \
    .withColumn("dst", F.col("dst").cast("string")) 

# OR `.selectExpr("CAST(src AS STRING) AS src", "CAST(dst AS STRING) AS dst", "* EXCEPT(src, dst)")`

g = GraphFrame(vertices_clean, edges_clean)

In [4]:
top_10 = g.inDegrees.sort("inDegree", ascending=False).limit(10)
top_10 = top_10.join(
    g.vertices.select("id", "name"),
    "id"
)
top_10.show()

+----+--------+--------------------+
|  id|inDegree|                name|
+----+--------+--------------------+
| 340|     493|Frankfurt am Main...|
| 507|     522|London Heathrow A...|
| 580|     450|Amsterdam Airport...|
|1382|     517|Charles de Gaulle...|
|3364|     534|Beijing Capital I...|
|3484|     498|Los Angeles Inter...|
|3670|     467|Dallas Fort Worth...|
|3682|     911|Hartsfield Jackso...|
|3797|     455|John F Kennedy In...|
|3830|     550|Chicago O'Hare In...|
+----+--------+--------------------+



In [5]:
# They ask to use Motifs, however using breath First algorithm or similar options would be more optimized!

g.find("(Turin)<-[]-(Any)") \
    .filter("Turin.id=1526") \
    .count()

42

In [6]:
# NOTE: `.filter("NOT (e1.src = e2.src AND e1.dst = e2.dst)")` would be reduntant, this would only remove self loops in case Any1=Any2=Turin, which is already solved above

g.find("(Turin)<-[e1]-(Any1); (Any1)<-[e2]-(Any2)") \
    .filter("Turin.id=1526") \
    .filter("Any1.id!=Turin.id AND Any2.id!=Turin.id AND Any1.id != Any2.id") \
    .count()

7684

In [7]:
g.find("(Turin)<-[]-(Any1); (Any1)<-[]-(Any2); (Any2)<-[]-(Any3)") \
    .filter("Turin.id=1526") \
    .filter("Any1.id!=Turin.id AND Any2.id!=Turin.id AND Any1.id != Any2.id") \
    .filter("Any3.id!=Turin.id AND Any3.id!=Any1.id AND Any3.id != Any2.id") \
    .count()

1252143

In [8]:
pathsToTurin = g.shortestPaths(landmarks=["1526"]) \
    .selectExpr("*", "distances['1526'] AS distance_to_turin") \
    .filter("distance_to_turin IS NOT NULL")

26/05/25 18:00:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/25 18:00:31 WARN BlockManager: Block rdd_170_0 already exists on this machine; not re-adding it
26/05/25 18:00:32 WARN ShortestPaths: Returned DataFrame is persistent and materialized!


In [9]:
max_distance = pathsToTurin.selectExpr("MAX(distance_to_turin)").collect()[0][0] # collect() is fine here since MAX() will always return just one row

In [10]:
pathsToTurin \
    .filter(f"distance_to_turin={max_distance}") \
    .select("name", "city", "country", "distances") \
    .show()

+-----------------+---------+-------+-----------+
|             name|     city|country|  distances|
+-----------------+---------+-------+-----------+
|Peawanuck Airport|Peawanuck| Canada|{1526 -> 8}|
+-----------------+---------+-------+-----------+



In [11]:
spark.sparkContext.setCheckpointDir("/tmp/spark-checkpoints")
cc_df = g.connectedComponents()

26/05/25 18:00:32 WARN ConnectedComponents: Algorithm 'graphframes' is deprecated and will be removed in a future release. Using 'two_phase' instead.
26/05/25 18:00:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/25 18:00:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/25 18:00:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/25 18:00:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/25 18:00:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/25 18:00:38 WARN MemoryManager: Total allocation exceeds 95.00% (1

In [12]:
large_components = cc_df.groupBy("component").count().filter("count(1)>=2")
large_components.count()

7